# MMMU Correct-CoT Generation & Validation — Direct Parquet / Range-Read Fix

This version fixes the Hugging Face `Dataset Viewer /rows` HTTP 500 problem.

## Retrieval design

The notebook **does not use Dataset Viewer `/rows` at all**.

For each requested MMMU ID it:

1. parses `{split}_{subject}_{1-based index}`;
2. lists the *real* Parquet shard names in the pinned `MMMU/MMMU` repository;
3. opens those files through Hugging Face's `hf://` filesystem at the pinned commit;
4. reads only Parquet metadata first to locate the exact shard and row group;
5. reads only the row group containing the requested sample;
6. checks that the recovered `row["id"]` is **exactly** the requested ID;
7. caches the exact recovered row and its image bytes locally;
8. reuses that cache on resume.

This avoids the broken Dataset Viewer mapping that was looking for stale paths such as
`Accounting/dev/0000.parquet`.

The two-stage Gemini pipeline is unchanged:
- Call 1: independent multimodal solving + structured CoT. No gold answer and no explanation.
- Wrong Call-1 answer: reject immediately; no Call 2.
- Correct Call-1 answer: Call 2 validates the generated CoT.
- If an original MMMU explanation exists, Call 2 uses it only as validation evidence.
- Accepted CoTs retain atomic `Step N:` formatting and `complexity_n_steps`.


## v3 API compatibility fix
Cell 1 now checks the installed `google-genai` package version. If it is
missing or older than 2.0.0, only `google-genai` is upgraded to a compatible
2.x release. Kaggle's preinstalled pandas/requests/pillow/fsspec stack is not
upgraded.


## v4 — Gemma 4 recommended generation + explicit thinking

This version changes the inference configuration and therefore uses a NEW
output/checkpoint directory so that previous runs are never mixed with this run.

### Call 1 — CoT solver
- model: `gemma-4-31b-it`
- thinking: `high` (explicitly enabled)
- temperature: `1.0`
- top_p: `0.95`
- top_k: `64`
- seed: `42`
- max output tokens: `4096`

### Call 2 — rationale verifier
- model: `gemma-4-31b-it`
- thinking: `high` (explicitly enabled)
- temperature: `0.0`
- top_p: `1.0`
- top_k: `64`
- seed: `42`
- max output tokens: `2048`

### HTTP robustness
- Google GenAI client timeout: `900000 ms` (15 minutes)
- transient retry/checkpoint logic remains enabled.

The CoT-generation and validation prompts are unchanged.


## v5 — Session-safe resume from a previous Kaggle Version

This version preserves the exact v4 experimental configuration and adds
strict recovery from a previous `cot_pipeline_all_results.csv`.

On a fresh Kaggle session it:

1. Finds the same 600-ID input file.
2. Searches `/kaggle/input` for a previous `cot_pipeline_all_results.csv`.
3. Validates that the checkpoint has:
   - exactly 600 unique requested IDs,
   - exactly the same ID set and input SHA256,
   - the same solver/verifier model and decoding configuration,
   - a single pinned MMMU dataset revision.
4. Reuses the checkpoint's exact dataset revision.
5. Copies the validated checkpoint into the new `/kaggle/working` output directory.
6. Regenerates the derived accepted-only/statistics files from checkpoint state.
7. Skips terminal samples.
8. Resumes retryable samples from the correct stage:
   - Stage-1 errors/incomplete -> retry Call 1.
   - Stage-2 errors -> keep the successful Call 1 and retry only Call 2.
   - untouched rows -> start Call 1 normally.

No accepted/rejected terminal sample is re-inferred.


## v6 — Explicit previous-output-as-input continuation

This version is designed for a NEW Kaggle session after a previous run stopped.

Inputs:
1. The SAME 600-ID TXT file used by the original run.
2. The previous run's `cot_pipeline_all_results.csv`.

Behavior:
- The previous CSV is READ-ONLY source data under `/kaggle/input`.
- It is strictly validated against the current 600-ID list and the exact
  experimental configuration.
- It is copied into `/kaggle/working` as the active cumulative checkpoint.
- All previous rows are preserved.
- Terminal rows are never inferred again.
- Retryable rows resume from their correct stage.
- New results UPDATE the corresponding row in the cumulative checkpoint.
- `accepted_only`, subject statistics, summary, and metadata are regenerated
  from the full cumulative state, so they contain old + new progress.
- Writes use atomic replacement where possible: a completed old checkpoint is
  not intentionally overwritten until the new complete CSV has been written.

Important:
Kaggle `/kaggle/input` is read-only. Therefore the notebook cannot literally
modify the uploaded old file in place. Instead it creates a cumulative copy
under `/kaggle/working` with the SAME output filenames. Download the newest
working/output files after each Version and use the newest
`cot_pipeline_all_results.csv` as the checkpoint for any later continuation.


In [1]:
# Cell 1 — Kaggle-safe dependency bootstrap with google-genai version check
#
# This cell intentionally DOES NOT upgrade Kaggle's scientific stack
# (pandas, requests, pillow, fsspec, etc.).
#
# The Interactions API requires google-genai >= 2.0.0.
# We therefore inspect the INSTALLED VERSION, not merely whether the
# module exists.

import importlib.util
import importlib.metadata as importlib_metadata
import subprocess
import sys

def installed_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None

def version_tuple(v):
    """
    Minimal numeric comparison adequate for normal google-genai releases
    such as 1.66.0, 2.0.0, 2.21.0.
    """
    if v is None:
        return None
    parts = []
    for x in str(v).split("."):
        digits = "".join(ch for ch in x if ch.isdigit())
        parts.append(int(digits) if digits else 0)
        if len(parts) == 3:
            break
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)

installed_google_genai = installed_version("google-genai")
print("Installed google-genai before bootstrap:", installed_google_genai)

need_google_genai_upgrade = (
    installed_google_genai is None
    or version_tuple(installed_google_genai) < (2, 0, 0)
)

packages_to_install = []

if need_google_genai_upgrade:
    # Pin to the known-compatible major version rather than allowing an
    # uncontrolled future major upgrade.
    packages_to_install.append("google-genai>=2.0.0,<3.0.0")

# These packages are required by this notebook, but are installed ONLY
# when completely absent. Existing Kaggle versions are preserved.
OPTIONAL_REQUIREMENTS = [
    ("huggingface_hub", "huggingface_hub"),
    ("pyarrow", "pyarrow"),
    ("fsspec", "fsspec"),
]

for module_name, package_spec in OPTIONAL_REQUIREMENTS:
    if importlib.util.find_spec(module_name) is None:
        packages_to_install.append(package_spec)

if packages_to_install:
    print("Installing/upgrading only:", packages_to_install)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *packages_to_install,
        ]
    )
else:
    print("Required dependencies already satisfy notebook requirements; pip skipped.")

final_google_genai = installed_version("google-genai")
print("Installed google-genai after bootstrap:", final_google_genai)

if final_google_genai is None or version_tuple(final_google_genai) < (2, 0, 0):
    raise RuntimeError(
        "google-genai >= 2.0.0 is required for the current Interactions API, "
        f"but detected version is {final_google_genai!r}."
    )

print("google-genai version check: OK (>= 2.0.0)")


Installed google-genai before bootstrap: 1.68.0
Installing/upgrading only: ['google-genai>=2.0.0,<3.0.0']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.57.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.22.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.22.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2

Installed google-genai after bootstrap: 2.22.0
google-genai version check: OK (>= 2.0.0)


In [2]:
# Cell 2 — Imports, configuration, Secrets, and reconnectable Google client

import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

import ast
import base64
import hashlib
import json
import re
import string
import time
import shutil
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image as PILImage, ImageOps
from datasets import load_dataset, Image
from huggingface_hub import HfApi

# ------------------------------------------------------------------
# Hard runtime guard for the current Google Interactions API
# ------------------------------------------------------------------
import importlib.metadata as importlib_metadata

def _numeric_version_tuple(v):
    parts = []
    for x in str(v).split("."):
        digits = "".join(ch for ch in x if ch.isdigit())
        parts.append(int(digits) if digits else 0)
        if len(parts) == 3:
            break
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)

_google_genai_version = importlib_metadata.version("google-genai")
if _numeric_version_tuple(_google_genai_version) < (2, 0, 0):
    raise RuntimeError(
        "google-genai is too old for the current Interactions API. "
        f"Detected {_google_genai_version}. Run Cell 1 first."
    )
print("Runtime google-genai:", _google_genai_version)

from google import genai
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

# ------------------------------------------------------------------
# MODEL
# ------------------------------------------------------------------
SOLVER_MODEL = "gemma-4-31b-it"
VERIFIER_MODEL = SOLVER_MODEL

# ------------------------------------------------------------------
# DATASET / POOL
# ------------------------------------------------------------------
DATASET_REPO = "MMMU/MMMU"
DATASET_REVISION = None  # resolved once to the current repository SHA
EXPECTED_N_IDS = 600
EXPECTED_N_SUBJECTS = 30
EXPECTED_PER_SUBJECT = 20

# ------------------------------------------------------------------
# DECODING
# ------------------------------------------------------------------
SOLVER_TEMPERATURE = 1.0
SOLVER_TOP_P = 0.95
SOLVER_TOP_K = 64
SOLVER_THINKING_LEVEL = "high"
SOLVER_MAX_OUTPUT_TOKENS = 4096

VERIFIER_TEMPERATURE = 0.0
VERIFIER_TOP_P = 1.0
VERIFIER_TOP_K = 64
VERIFIER_THINKING_LEVEL = "high"
VERIFIER_MAX_OUTPUT_TOKENS = 2048

MODEL_SEED = 42

assert SOLVER_THINKING_LEVEL in {"high", "minimal"}
assert VERIFIER_THINKING_LEVEL in {"high", "minimal"}
assert SOLVER_TOP_K >= 1 and VERIFIER_TOP_K >= 1

# ------------------------------------------------------------------
# IMAGE PREPROCESSING
# ------------------------------------------------------------------
COMPRESS_IMAGES_FOR_API = True
API_IMAGE_MAX_SIDE = 1280
API_IMAGE_TARGET_BYTES = 350_000
API_JPEG_QUALITY_START = 92
API_JPEG_QUALITY_MIN = 72

# ------------------------------------------------------------------
# API ROBUSTNESS
# ------------------------------------------------------------------
REQUEST_DELAY_SECONDS = 5.0
MAX_API_RETRIES = 4
INITIAL_BACKOFF_SECONDS = 15.0
MAX_BACKOFF_SECONDS = 120.0

# 15-minute client-side timeout.
# The Google SDK expects milliseconds.
API_HTTP_TIMEOUT_MS = 900_000

# ------------------------------------------------------------------
# SESSION-RESUME
# ------------------------------------------------------------------
# PREVIOUS RUN OUTPUT
# Recommended: upload/attach the previous run's cot_pipeline_all_results.csv
# as a Kaggle Input. Leave this None for automatic discovery.
#
# If automatic discovery reports more than one valid candidate, set this to
# the exact /kaggle/input/... path printed in the logs.
PREVIOUS_RESULTS_CSV = None

AUTO_DISCOVER_PREVIOUS_RESULTS = True

# Accept the canonical name and harmless download suffixes such as:
#   cot_pipeline_all_results.csv
#   cot_pipeline_all_results(1).csv
#   cot_pipeline_all_results_1.csv
PREVIOUS_RESULTS_GLOB = "cot_pipeline_all_results*.csv"

# ------------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------------
OUTPUT_DIR = Path(
    "/kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
HF_RECORDS_PATH = OUTPUT_DIR / "selected_hf_records_metadata.csv"
ALL_RESULTS_PATH = OUTPUT_DIR / "cot_pipeline_all_results.csv"
ACCEPTED_ONLY_PATH = OUTPUT_DIR / "cot_pipeline_accepted_only.csv"
SUBJECT_STATS_PATH = OUTPUT_DIR / "cot_pipeline_subject_stats.csv"
SUMMARY_PATH = OUTPUT_DIR / "cot_pipeline_summary.json"

# Separate exact-row/image cache. It is safe to reuse across reruns because
# every cached row is checked against both the requested ID and dataset SHA.
HF_ROW_CACHE_DIR = OUTPUT_DIR / "hf_exact_row_cache"
HF_ROW_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# REPRODUCIBILITY / PROMPT IDs
# ------------------------------------------------------------------
CALL1_PROMPT_VERSION = "solver_atomic_steps_v1"
CALL2_PROMPT_VERSION = "validator_explanation_consistency_v1"

# ------------------------------------------------------------------
# HF SETTINGS
# ------------------------------------------------------------------
# MMMU is public, so HF_TOKEN is optional.
HF_TOKEN_SECRET = "HF_TOKEN"
PARQUET_OPEN_FILE_CACHE = True
STRICT_EXACT_ID_VALIDATION = True

# ------------------------------------------------------------------
# Logging
# ------------------------------------------------------------------
def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# ------------------------------------------------------------------
# Kaggle Secrets
# ------------------------------------------------------------------
def get_secret(name: str, required: bool = True):
    """
    Read a secret from either:
      1) an environment variable, or
      2) Kaggle Secrets.

    `required=False` is used for optional HF_TOKEN.
    """
    value = os.environ.get(name)
    if value:
        return value

    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        # Optional secrets are allowed to be absent.
        pass

    if required:
        raise RuntimeError(
            f"{name} not found. Add a private Kaggle Secret named {name}."
        )
    return None

# ------------------------------------------------------------------
# Reconnectable Google client
# ------------------------------------------------------------------
# For the first run create a Kaggle Secret named exactly GEMINI_API_KEY.
# If quota is exhausted later, you may overwrite that same Kaggle Secret
# with a new key and rerun this cell, OR use a second alias such as
# GEMINI_API_KEY_2 and call connect_google_client("GEMINI_API_KEY_2").
ACTIVE_API_SECRET = "GEMINI_API_KEY"
client = None

def connect_google_client(secret_name=None):
    global client, ACTIVE_API_SECRET

    if secret_name is not None:
        ACTIVE_API_SECRET = str(secret_name)

    api_key = get_secret(ACTIVE_API_SECRET, required=True)
    client = genai.Client(api_key=api_key, http_options={"timeout": API_HTTP_TIMEOUT_MS})

    print("Connected with Kaggle Secret alias:", ACTIVE_API_SECRET)
    return client

# Connect once now.
connect_google_client(ACTIVE_API_SECRET)

print("Configuration loaded.")
print("Solver:", SOLVER_MODEL)
print(
    "  Call 1 config:",
    f"thinking={SOLVER_THINKING_LEVEL}",
    f"temperature={SOLVER_TEMPERATURE}",
    f"top_p={SOLVER_TOP_P}",
    f"top_k={SOLVER_TOP_K}",
    f"max_tokens={SOLVER_MAX_OUTPUT_TOKENS}",
    f"seed={MODEL_SEED}",
)
print("Verifier:", VERIFIER_MODEL)
print(
    "  Call 2 config:",
    f"thinking={VERIFIER_THINKING_LEVEL}",
    f"temperature={VERIFIER_TEMPERATURE}",
    f"top_p={VERIFIER_TOP_P}",
    f"top_k={VERIFIER_TOP_K}",
    f"max_tokens={VERIFIER_MAX_OUTPUT_TOKENS}",
    f"seed={MODEL_SEED}",
)
print("Google client HTTP timeout (ms):", API_HTTP_TIMEOUT_MS)
print("Outputs:", OUTPUT_DIR)
print("HF_TOKEN present:", bool(get_secret(HF_TOKEN_SECRET, required=False)))

# Optional diagnostics — useful when reproducing the experiment.
try:
    import importlib.metadata as importlib_metadata
    print("google-genai:", importlib_metadata.version("google-genai"))
    print("pandas:", importlib_metadata.version("pandas"))
    print("pyarrow:", importlib_metadata.version("pyarrow"))
    print("huggingface_hub:", importlib_metadata.version("huggingface_hub"))
    print("fsspec:", importlib_metadata.version("fsspec"))
except Exception as exc:
    print("Version diagnostic warning:", exc)


Runtime google-genai: 2.22.0
Connected with Kaggle Secret alias: GEMINI_API_KEY
Configuration loaded.
Solver: gemma-4-31b-it
  Call 1 config: thinking=high temperature=1.0 top_p=0.95 top_k=64 max_tokens=4096 seed=42
Verifier: gemma-4-31b-it
  Call 2 config: thinking=high temperature=0.0 top_p=1.0 top_k=64 max_tokens=2048 seed=42
Google client HTTP timeout (ms): 900000
Outputs: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4
HF_TOKEN present: False
google-genai: 2.22.0
pandas: 2.3.3
pyarrow: 24.0.0
huggingface_hub: 1.11.0
fsspec: 2025.3.0


In [3]:
# Cell 3 — Locate IDs, pin MMMU revision, and define DIRECT PARQUET lazy retrieval

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
SPLIT_NAMES = ("validation", "test", "dev")

def all_visible_files():
    out = {}
    for root in SEARCH_ROOTS:
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file():
                    out[str(p)] = p
    return list(out.values())

def parse_mmmu_id(qid):
    """
    Parse IDs such as:
      test_Accounting_22
      validation_Physics_30
      dev_Biology_3

    MMMU numeric suffixes are treated as 1-based split positions.
    We NEVER trust the position alone: recovered row['id'] must match qid.
    """
    qid = str(qid).strip()
    for split in SPLIT_NAMES:
        prefix = split + "_"
        if qid.startswith(prefix):
            rest = qid[len(prefix):]
            subject, idx = rest.rsplit("_", 1)
            if idx.isdigit():
                idx = int(idx)
                if idx < 1:
                    raise ValueError(f"MMMU index must be >= 1: {qid}")
                return {
                    "id": qid,
                    "split": split,
                    "subject": subject,
                    "index": idx,
                    "expected_offset": idx - 1,
                }
    raise ValueError(f"Bad MMMU ID: {qid!r}")

def read_ids(path):
    with Path(path).open("r", encoding="utf-8-sig") as f:
        return [x.strip() for x in f if x.strip()]

def valid_id_file(path):
    try:
        ids = read_ids(path)
        if len(ids) != EXPECTED_N_IDS or len(set(ids)) != EXPECTED_N_IDS:
            return False, None
        d = pd.DataFrame([parse_mmmu_id(x) for x in ids])
        counts = d["subject"].value_counts()
        ok = (
            d["subject"].nunique() == EXPECTED_N_SUBJECTS
            and (counts == EXPECTED_PER_SUBJECT).all()
        )
        return ok, ids
    except Exception:
        return False, None

def find_id_file():
    candidates = [
        p for p in all_visible_files()
        if p.suffix.lower() == ".txt"
        and all(k in p.name.lower() for k in ["correct", "cot", "similarity", "ids"])
    ]
    good = []
    for p in candidates:
        ok, ids = valid_id_file(p)
        if ok:
            good.append((p, ids))

    if not good:
        print("Candidate TXT files:")
        for p in candidates:
            try:
                x = read_ids(p)
                print(" ", p, "lines=", len(x), "unique=", len(set(x)))
            except Exception:
                print(" ", p)
        raise FileNotFoundError(
            "Valid 600-ID correct_cot_similarity_pool_ids*.txt not found."
        )

    ref = set(good[0][1])
    if any(set(ids) != ref for _, ids in good[1:]):
        raise RuntimeError("Multiple different valid ID files found; remove extras.")

    good.sort(key=lambda x: ("/kaggle/input/" not in str(x[0]), len(str(x[0]))))
    return good[0][0]

IDS_PATH = find_id_file()
REQUESTED_IDS = read_ids(IDS_PATH)

parsed_ids_df = pd.DataFrame([parse_mmmu_id(x) for x in REQUESTED_IDS])
parsed_ids_df.insert(0, "request_index", np.arange(1, len(parsed_ids_df) + 1))

assert len(REQUESTED_IDS) == EXPECTED_N_IDS
assert len(set(REQUESTED_IDS)) == EXPECTED_N_IDS
assert parsed_ids_df["subject"].nunique() == EXPECTED_N_SUBJECTS
assert (parsed_ids_df["subject"].value_counts() == EXPECTED_PER_SUBJECT).all()

PARSED_BY_ID = {r["id"]: r for r in parsed_ids_df.to_dict("records")}

input_ids_sha256 = hashlib.sha256(
    ("\n".join(REQUESTED_IDS) + "\n").encode("utf-8")
).hexdigest()

# ------------------------------------------------------------------
# STRICT EXTERNAL CHECKPOINT DISCOVERY / VALIDATION
# ------------------------------------------------------------------
def _checkpoint_candidates():
    """
    Discover ONLY previous output files supplied as Kaggle Inputs.
    Never mistake the new /kaggle/working checkpoint for an external source.
    """
    if PREVIOUS_RESULTS_CSV:
        return [Path(PREVIOUS_RESULTS_CSV)]

    if not AUTO_DISCOVER_PREVIOUS_RESULTS:
        return []

    root = Path("/kaggle/input")
    if not root.exists():
        return []

    try:
        found = list(root.rglob(PREVIOUS_RESULTS_GLOB))
    except Exception:
        found = []

    return sorted(
        {p for p in found if p.is_file()},
        key=lambda p: str(p)
    )


def _scalar_nonempty_unique(series):
    vals = []
    for x in series.dropna().tolist():
        s = str(x).strip()
        if s and s.lower() != "nan":
            vals.append(s)
    return sorted(set(vals))


def _same_numeric(series, expected, tol=1e-12):
    vals = pd.to_numeric(series, errors="coerce").dropna().unique()
    if len(vals) == 0:
        return False
    return all(abs(float(v) - float(expected)) <= tol for v in vals)


def validate_resume_checkpoint(path):
    path = Path(path)
    info = {"path": str(path), "valid": False, "reason": None}
    if not path.exists() or not path.is_file():
        info["reason"] = "path does not exist"
        return info

    try:
        old = pd.read_csv(path)
    except Exception as exc:
        info["reason"] = f"CSV read failed: {type(exc).__name__}: {exc}"
        return info

    required = {
        "request_index","question_id","pipeline_status",
        "dataset_revision","input_id_sha256",
        "call1_model","call1_temperature","call1_top_p","call1_top_k",
        "call1_thinking_level","call1_seed","call1_max_output_tokens",
        "call2_model","call2_temperature","call2_top_p","call2_top_k",
        "call2_thinking_level","call2_seed","call2_max_output_tokens",
    }
    missing = sorted(required - set(old.columns))
    if missing:
        info["reason"] = f"missing required columns: {missing}"
        return info

    ids = old["question_id"].astype(str).tolist()
    if len(old) != EXPECTED_N_IDS:
        info["reason"] = f"expected {EXPECTED_N_IDS} rows, got {len(old)}"
        return info
    if len(set(ids)) != EXPECTED_N_IDS:
        info["reason"] = "checkpoint question_id values are not unique"
        return info
    if set(ids) != set(REQUESTED_IDS):
        info["reason"] = "checkpoint ID set differs from current 600-ID input"
        return info

    idx = pd.to_numeric(old["request_index"], errors="coerce")
    if idx.isna().any() or set(idx.astype(int)) != set(range(1, EXPECTED_N_IDS + 1)):
        info["reason"] = "request_index is not exactly 1..600"
        return info

    ordered_ids = (
        old.assign(_idx=idx.astype(int))
           .sort_values("_idx")["question_id"]
           .astype(str)
           .tolist()
    )
    if ordered_ids != REQUESTED_IDS:
        info["reason"] = (
            "checkpoint request order differs from the current ID TXT. "
            "Use the exact same ID file as the original run."
        )
        return info

    sha_vals = _scalar_nonempty_unique(old["input_id_sha256"])
    if sha_vals != [input_ids_sha256]:
        info["reason"] = (
            f"input ID SHA mismatch: checkpoint={sha_vals}, current={input_ids_sha256}"
        )
        return info

    rev_vals = _scalar_nonempty_unique(old["dataset_revision"])
    if len(rev_vals) != 1:
        info["reason"] = f"expected one pinned dataset revision, found {rev_vals}"
        return info

    # Exact experimental-configuration guard.
    text_checks = {
        "call1_model": SOLVER_MODEL,
        "call1_thinking_level": SOLVER_THINKING_LEVEL,
        "call2_model": VERIFIER_MODEL,
        "call2_thinking_level": VERIFIER_THINKING_LEVEL,
    }
    for col, expected in text_checks.items():
        vals = _scalar_nonempty_unique(old[col])
        if vals != [str(expected)]:
            info["reason"] = f"{col} mismatch: {vals} != {[str(expected)]}"
            return info

    numeric_checks = {
        "call1_temperature": SOLVER_TEMPERATURE,
        "call1_top_p": SOLVER_TOP_P,
        "call1_top_k": SOLVER_TOP_K,
        "call1_seed": MODEL_SEED,
        "call1_max_output_tokens": SOLVER_MAX_OUTPUT_TOKENS,
        "call2_temperature": VERIFIER_TEMPERATURE,
        "call2_top_p": VERIFIER_TOP_P,
        "call2_top_k": VERIFIER_TOP_K,
        "call2_seed": MODEL_SEED,
        "call2_max_output_tokens": VERIFIER_MAX_OUTPUT_TOKENS,
    }
    for col, expected in numeric_checks.items():
        if not _same_numeric(old[col], expected):
            vals = pd.to_numeric(old[col], errors="coerce").dropna().unique().tolist()
            info["reason"] = f"{col} mismatch: {vals} != {expected}"
            return info

    status_counts = {
        str(k): int(v)
        for k, v in old["pipeline_status"].fillna("NaN").value_counts().to_dict().items()
    }

    info.update({
        "valid": True,
        "reason": "OK",
        "dataset_revision": rev_vals[0],
        "rows": int(len(old)),
        "status_counts": status_counts,
    })
    return info


_resume_infos = [validate_resume_checkpoint(p) for p in _checkpoint_candidates()]
_valid_resume_infos = [x for x in _resume_infos if x["valid"]]

if _resume_infos:
    print("Previous-output checkpoint candidates:")
    for x in _resume_infos:
        print(" ", x["path"], "| valid=", x["valid"], "|", x["reason"])

if len(_valid_resume_infos) > 1:
    raise RuntimeError(
        "Multiple valid resume checkpoints were found. "
        "Set RESUME_CHECKPOINT_OVERRIDE to the exact intended path."
    )

RESUME_CHECKPOINT_SOURCE = None
if len(_valid_resume_infos) == 1:
    RESUME_CHECKPOINT_SOURCE = Path(_valid_resume_infos[0]["path"])
    checkpoint_revision = _valid_resume_infos[0]["dataset_revision"]

    if DATASET_REVISION is not None and str(DATASET_REVISION) != checkpoint_revision:
        raise RuntimeError(
            f"Configured DATASET_REVISION={DATASET_REVISION} does not match "
            f"checkpoint revision={checkpoint_revision}"
        )
    DATASET_REVISION = checkpoint_revision

    print("VALIDATED PREVIOUS OUTPUT:", RESUME_CHECKPOINT_SOURCE)
    print("Checkpoint status counts:", _valid_resume_infos[0]["status_counts"])
    print("Dataset revision restored from checkpoint:", DATASET_REVISION)

# If no checkpoint exists, pin the current repository state as in v4.
hf_api = HfApi(token=get_secret(HF_TOKEN_SECRET, required=False))
if DATASET_REVISION is None:
    DATASET_REVISION = str(hf_api.dataset_info(DATASET_REPO).sha)

HF_TOKEN = get_secret(HF_TOKEN_SECRET, required=False)

print("ID file:", IDS_PATH)
print("IDs:", len(REQUESTED_IDS), "| subjects:", parsed_ids_df["subject"].nunique())
print("Pinned MMMU revision:", DATASET_REVISION)
print("ID SHA256:", input_ids_sha256)
print("Retrieval backend: pinned HF repository Parquet files via hf:// range reads")
print("Dataset Viewer /rows: DISABLED")
display(parsed_ids_df["split"].value_counts().rename_axis("split").reset_index(name="samples"))

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "retrieval_backend_primary": "hf_repository_parquet_range_read",
    "dataset_viewer_used": False,
    "exact_id_validation": True,
    "numeric_id_suffix_interpretation": "1_based_split_position",
    "input_id_file": str(IDS_PATH),
    "input_id_sha256": input_ids_sha256,
    "requested_n": len(REQUESTED_IDS),
    "requested_subjects": int(parsed_ids_df["subject"].nunique()),
    "requested_per_subject": EXPECTED_PER_SUBJECT,
    "solver_model": SOLVER_MODEL,
    "verifier_model": VERIFIER_MODEL,
    "solver_temperature": SOLVER_TEMPERATURE,
    "solver_top_p": SOLVER_TOP_P,
    "solver_top_k": SOLVER_TOP_K,
    "solver_thinking_level": SOLVER_THINKING_LEVEL,
    "solver_max_output_tokens": SOLVER_MAX_OUTPUT_TOKENS,
    "verifier_temperature": VERIFIER_TEMPERATURE,
    "verifier_top_p": VERIFIER_TOP_P,
    "verifier_top_k": VERIFIER_TOP_K,
    "verifier_thinking_level": VERIFIER_THINKING_LEVEL,
    "verifier_max_output_tokens": VERIFIER_MAX_OUTPUT_TOKENS,
    "seed": MODEL_SEED,
    "api_http_timeout_ms": API_HTTP_TIMEOUT_MS,
    "call1_prompt_version": CALL1_PROMPT_VERSION,
    "call2_prompt_version": CALL2_PROMPT_VERSION,
    "call1_gold_visible": False,
    "call1_explanation_visible": False,
    "call2_gold_visible": True,
    "call2_explanation_used_if_available": True,
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

def parse_options(value):
    if isinstance(value, (list, tuple)):
        return [str(x) for x in value]
    if isinstance(value, str):
        x = ast.literal_eval(value)
        if isinstance(x, (list, tuple)):
            return [str(v) for v in x]
    raise TypeError(f"Unsupported options: {type(value)}")

def meaningful_explanation(value):
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    if text.lower() in {
        "", "?", "n/a", "na", "none", "null", "nan", "unknown",
        "not available", "no explanation", "no explanation available"
    }:
        return False
    return len(re.sub(r"[^A-Za-z0-9]+", "", text)) >= 3

# ------------------------------------------------------------------
# Direct repo file discovery
# ------------------------------------------------------------------
_REPO_FILES = None
_SPLIT_SHARD_CACHE = {}

def _all_repo_files():
    global _REPO_FILES
    if _REPO_FILES is None:
        log("Listing real MMMU repository files at pinned revision...")
        _REPO_FILES = hf_api.list_repo_files(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            revision=DATASET_REVISION,
        )
        log(f"Repository file listing ready: {len(_REPO_FILES)} files")
    return _REPO_FILES

def _split_shards(subject, split):
    """
    Return the REAL source files, e.g.
      Accounting/test-00000-of-00001.parquet
    never stale Dataset Viewer paths like Accounting/test/0000.parquet.
    """
    key = (subject, split)
    if key in _SPLIT_SHARD_CACHE:
        return _SPLIT_SHARD_CACHE[key]

    pat = re.compile(
        rf"^{re.escape(subject)}/{re.escape(split)}-\d{{5}}-of-\d{{5}}\.parquet$"
    )
    files = sorted(p for p in _all_repo_files() if pat.match(p))
    if not files:
        raise FileNotFoundError(
            f"No real Parquet files found for {subject}/{split} "
            f"at revision {DATASET_REVISION}"
        )
    _SPLIT_SHARD_CACHE[key] = files
    return files

def _hf_uri(path_in_repo):
    return (
        f"hf://datasets/{DATASET_REPO}@{DATASET_REVISION}/"
        f"{path_in_repo}"
    )

# ------------------------------------------------------------------
# Parquet footer / open-file caches
# ------------------------------------------------------------------
_PARQUET_FILE_CACHE = {}
_SHARD_META_CACHE = {}
_SPLIT_LAYOUT_CACHE = {}

def _open_parquet(path_in_repo):
    """
    PyArrow + hf:// performs seek/range-based remote reads.
    Keeping ParquetFile open also prevents repeated footer setup in a run.
    """
    if PARQUET_OPEN_FILE_CACHE and path_in_repo in _PARQUET_FILE_CACHE:
        return _PARQUET_FILE_CACHE[path_in_repo]

    uri = _hf_uri(path_in_repo)
    log(f"Opening Parquet metadata: {path_in_repo}")
    pf = pq.ParquetFile(uri)

    if PARQUET_OPEN_FILE_CACHE:
        _PARQUET_FILE_CACHE[path_in_repo] = pf
    return pf

def _shard_meta(path_in_repo):
    if path_in_repo in _SHARD_META_CACHE:
        return _SHARD_META_CACHE[path_in_repo]

    pf = _open_parquet(path_in_repo)
    meta = {
        "path": path_in_repo,
        "num_rows": int(pf.metadata.num_rows),
        "num_row_groups": int(pf.metadata.num_row_groups),
        "row_group_rows": [
            int(pf.metadata.row_group(i).num_rows)
            for i in range(pf.metadata.num_row_groups)
        ],
        "schema_names": list(pf.schema_arrow.names),
    }
    _SHARD_META_CACHE[path_in_repo] = meta
    return meta

def _split_layout(subject, split):
    """
    Build only lightweight footer metadata:
    global split row range -> real shard.
    """
    key = (subject, split)
    if key in _SPLIT_LAYOUT_CACHE:
        return _SPLIT_LAYOUT_CACHE[key]

    layout = []
    start = 0
    for path in _split_shards(subject, split):
        m = _shard_meta(path)
        end = start + m["num_rows"]
        layout.append({
            **m,
            "global_start": int(start),
            "global_end_exclusive": int(end),
        })
        start = end

    _SPLIT_LAYOUT_CACHE[key] = layout
    log(
        f"Layout {subject}/{split}: "
        f"{sum(x['num_rows'] for x in layout)} rows | "
        f"{len(layout)} shard(s) | "
        f"{sum(x['num_row_groups'] for x in layout)} row group(s)"
    )
    return layout

def _locate_shard_and_row_group(subject, split, global_offset):
    layout = _split_layout(subject, split)

    shard = None
    for x in layout:
        if x["global_start"] <= global_offset < x["global_end_exclusive"]:
            shard = x
            break

    if shard is None:
        total = layout[-1]["global_end_exclusive"] if layout else 0
        raise IndexError(
            f"{subject}/{split}: offset {global_offset} outside 0..{total-1}"
        )

    local_in_shard = int(global_offset - shard["global_start"])

    rg_start = 0
    rg_idx = None
    local_in_rg = None
    for i, nrows in enumerate(shard["row_group_rows"]):
        rg_end = rg_start + nrows
        if rg_start <= local_in_shard < rg_end:
            rg_idx = i
            local_in_rg = local_in_shard - rg_start
            break
        rg_start = rg_end

    if rg_idx is None:
        raise RuntimeError(
            f"Could not locate row group for {subject}/{split} offset={global_offset}"
        )

    return shard, int(rg_idx), int(local_in_rg)

# ------------------------------------------------------------------
# Exact row read
# ------------------------------------------------------------------
def _read_exact_parquet_row(qid):
    p = PARSED_BY_ID[qid]
    global_offset = int(p["expected_offset"])

    shard, rg_idx, local_in_rg = _locate_shard_and_row_group(
        p["subject"], p["split"], global_offset
    )
    pf = _open_parquet(shard["path"])

    schema_names = shard["schema_names"]
    image_columns = [
        c for c in schema_names if re.fullmatch(r"image_\d+", str(c))
    ]
    nonimage_columns = [c for c in schema_names if c not in image_columns]

    # First read only textual/metadata columns. This lets us validate the ID
    # BEFORE any image column is materialized.
    meta_table = pf.read_row_group(rg_idx, columns=nonimage_columns)
    if local_in_rg >= meta_table.num_rows:
        raise RuntimeError(
            f"{qid}: local row {local_in_rg} >= row-group size {meta_table.num_rows}"
        )

    meta_row = meta_table.slice(local_in_rg, 1).to_pylist()[0]
    got_id = str(meta_row.get("id", "")).strip()

    if STRICT_EXACT_ID_VALIDATION and got_id != qid:
        raise RuntimeError(
            f"EXACT-ID MISMATCH for {qid}: recovered {got_id!r}. "
            f"Refusing to send any data to Gemini."
        )

    # Only now read image columns from the same row group.
    # Parquet remains columnar/range-read: no unrelated textual columns are reread.
    row = dict(meta_row)
    if image_columns:
        image_table = pf.read_row_group(rg_idx, columns=image_columns)
        image_row = image_table.slice(local_in_rg, 1).to_pylist()[0]
        row.update(image_row)

    backend = "hf_repo_parquet_row_group"
    location = {
        "source_file": shard["path"],
        "row_group_idx": rg_idx,
        "local_row_in_group": local_in_rg,
        "global_split_offset": global_offset,
        "num_row_groups_in_shard": shard["num_row_groups"],
    }
    return row, location, backend

# ------------------------------------------------------------------
# Exact per-sample disk cache
# ------------------------------------------------------------------
_ITEM_CACHE = {}

def _qid_cache_dir(qid):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", qid)
    d = HF_ROW_CACHE_DIR / safe
    d.mkdir(parents=True, exist_ok=True)
    return d

def _pil_to_png_bytes(img):
    b = BytesIO()
    img.save(b, format="PNG")
    return b.getvalue()

def _raw_image_bytes(value):
    if value is None:
        return None

    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return bytes(value["bytes"])
        if value.get("path"):
            return Path(value["path"]).read_bytes()

    if isinstance(value, (bytes, bytearray)):
        return bytes(value)

    if hasattr(value, "save"):
        return _pil_to_png_bytes(value)

    raise TypeError(f"Unsupported image value: {type(value)}")

def _validate_image_bytes(raw, qid, col):
    if not raw:
        raise RuntimeError(f"{qid} {col}: empty image bytes")
    try:
        with PILImage.open(BytesIO(raw)) as im:
            im.verify()
    except Exception as exc:
        raise RuntimeError(f"{qid} {col}: invalid image bytes: {exc}")

def _write_item_cache(qid, row, location, backend):
    d = _qid_cache_dir(qid)

    nonimage = {}
    image_columns = []
    image_sha256 = {}
    total_image_bytes = 0

    for k, v in row.items():
        if re.fullmatch(r"image_\d+", str(k)):
            if v is None:
                continue
            raw = _raw_image_bytes(v)
            if raw is None:
                continue

            _validate_image_bytes(raw, qid, k)
            image_columns.append(k)

            image_path = d / f"{k}.bin"
            image_path.write_bytes(raw)
            image_sha256[k] = hashlib.sha256(raw).hexdigest()
            total_image_bytes += len(raw)
        else:
            nonimage[k] = v

    cache_payload = {
        "question_id": qid,
        "backend": backend,
        "dataset_repo": DATASET_REPO,
        "dataset_revision": DATASET_REVISION,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "location": location,
        "nonimage_record": nonimage,
        "image_columns": image_columns,
        "image_sha256": image_sha256,
        "total_image_bytes": int(total_image_bytes),
    }

    (d / "record.json").write_text(
        json.dumps(cache_payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8"
    )
    return cache_payload

def _load_cached_item(qid):
    d = _qid_cache_dir(qid)
    record_path = d / "record.json"
    if not record_path.exists():
        return None

    try:
        payload = json.loads(record_path.read_text(encoding="utf-8"))
        if payload.get("question_id") != qid:
            return None
        if payload.get("dataset_revision") != DATASET_REVISION:
            return None

        item = dict(payload.get("nonimage_record", {}))
        for col in payload.get("image_columns", []):
            image_path = d / f"{col}.bin"
            if not image_path.exists():
                return None
            item[col] = {"path": str(image_path)}

        p = PARSED_BY_ID[qid]
        item["_source_config"] = p["subject"]
        item["_source_split"] = p["split"]
        item["_retrieval_backend"] = payload.get("backend")
        item["_retrieved_utc"] = payload.get("retrieved_utc")
        item["_image_sha256"] = payload.get("image_sha256", {})
        item["_total_image_bytes"] = payload.get("total_image_bytes", 0)
        item["_parquet_location"] = payload.get("location", {})

        if str(item.get("id", "")).strip() != qid:
            return None

        return item
    except Exception:
        return None

def _validate_item(qid, item):
    p = PARSED_BY_ID[qid]

    got = str(item.get("id", "")).strip()
    if got != qid:
        raise RuntimeError(
            f"{qid}: exact-ID validation failed; recovered {got!r}"
        )

    opts = parse_options(item["options"])
    gold = str(item.get("answer", "")).strip().upper()

    if gold not in set(string.ascii_uppercase[:len(opts)]):
        raise ValueError(
            f"{qid}: invalid gold={gold!r} for {len(opts)} options"
        )

    if item["_source_config"] != p["subject"]:
        raise AssertionError((qid, item["_source_config"], p["subject"]))
    if item["_source_split"] != p["split"]:
        raise AssertionError((qid, item["_source_split"], p["split"]))

def _metadata_from_item(qid, item):
    p = PARSED_BY_ID[qid]
    exp = str(item.get("explanation") or "").strip()
    nonimage = {
        k: v for k, v in item.items()
        if not re.fullmatch(r"image_\d+", str(k))
        and not str(k).startswith("_")
    }
    image_columns = [
        f"image_{i}" for i in range(1, 8)
        if item.get(f"image_{i}") is not None
    ]
    loc = item.get("_parquet_location", {}) or {}

    return {
        "request_index": int(p["request_index"]),
        "question_id": qid,
        "subject": p["subject"],
        "source_split": p["split"],
        "retrieved": True,
        "retrieval_backend": str(item.get("_retrieval_backend") or ""),
        "hf_source_file": str(loc.get("source_file") or ""),
        "hf_row_group_idx": loc.get("row_group_idx"),
        "hf_local_row_in_group": loc.get("local_row_in_group"),
        "hf_global_split_offset": loc.get("global_split_offset"),
        "hf_retrieved_utc": str(item.get("_retrieved_utc") or ""),
        "question": str(item.get("question") or ""),
        "options_json": json.dumps(parse_options(item["options"]), ensure_ascii=False),
        "gold_answer": str(item.get("answer") or "").strip().upper(),
        "original_explanation": exp,
        "has_explanation": meaningful_explanation(exp),
        "topic_difficulty": str(item.get("topic_difficulty") or ""),
        "subfield": str(item.get("subfield") or ""),
        "image_count": len(image_columns),
        "image_sha256_json": json.dumps(
            item.get("_image_sha256", {}), ensure_ascii=False, sort_keys=True
        ),
        "hf_image_bytes": int(item.get("_total_image_bytes") or 0),
        "hf_nonimage_record_json": json.dumps(
            nonimage, ensure_ascii=False, default=str
        ),
    }

HF_META_COLUMNS = [
    "request_index", "question_id", "subject", "source_split",
    "retrieved", "retrieval_backend",
    "hf_source_file", "hf_row_group_idx", "hf_local_row_in_group",
    "hf_global_split_offset", "hf_retrieved_utc",
    "question", "options_json", "gold_answer",
    "original_explanation", "has_explanation",
    "topic_difficulty", "subfield", "image_count",
    "image_sha256_json", "hf_image_bytes", "hf_nonimage_record_json",
]

def _initial_hf_meta_df():
    base = []
    for r in parsed_ids_df.to_dict("records"):
        base.append({
            "request_index": int(r["request_index"]),
            "question_id": r["id"],
            "subject": r["subject"],
            "source_split": r["split"],
            "retrieved": False,
        })
    df = pd.DataFrame(base)
    for c in HF_META_COLUMNS:
        if c not in df:
            df[c] = None
    return df[HF_META_COLUMNS]

hf_meta_df = _initial_hf_meta_df()

if HF_RECORDS_PATH.exists():
    try:
        old = pd.read_csv(HF_RECORDS_PATH)
        old = old[old["question_id"].astype(str).isin(REQUESTED_IDS)].copy()
        for _, rr in old.iterrows():
            qid = str(rr["question_id"])
            mask = hf_meta_df["question_id"].astype(str).eq(qid)
            for c in HF_META_COLUMNS:
                if c in rr.index:
                    v = rr[c]
                    try:
                        if pd.isna(v):
                            continue
                    except Exception:
                        pass
                    hf_meta_df.loc[mask, c] = v
    except Exception as exc:
        log(f"Could not reuse old HF metadata CSV: {exc}")

hf_meta_df.to_csv(HF_RECORDS_PATH, index=False)

def _upsert_hf_metadata(qid, item):
    global hf_meta_df
    m = _metadata_from_item(qid, item)
    mask = hf_meta_df["question_id"].astype(str).eq(qid)

    if not mask.any():
        hf_meta_df = pd.concat(
            [hf_meta_df, pd.DataFrame([m])],
            ignore_index=True
        )
    else:
        for k, v in m.items():
            hf_meta_df.loc[mask, k] = v

    hf_meta_df = hf_meta_df.sort_values("request_index").reset_index(drop=True)
    hf_meta_df.to_csv(HF_RECORDS_PATH, index=False)

def get_item(qid, verbose=True):
    """
    Retrieve exactly one requested MMMU sample lazily from the REAL repository
    Parquet files. Dataset Viewer is never called.
    """
    if qid in _ITEM_CACHE:
        return _ITEM_CACHE[qid]

    cached = _load_cached_item(qid)
    if cached is not None:
        _validate_item(qid, cached)
        _ITEM_CACHE[qid] = cached
        _upsert_hf_metadata(qid, cached)
        if verbose:
            loc = cached.get("_parquet_location", {}) or {}
            log(
                f"HF cache hit: {qid} | "
                f"{loc.get('source_file','?')} | rg={loc.get('row_group_idx','?')}"
            )
        return cached

    p = PARSED_BY_ID[qid]
    if verbose:
        log(
            f"HF Parquet exact fetch: {qid} | "
            f"{p['subject']}/{p['split']} | global_offset={p['expected_offset']}"
        )

    row, location, backend = _read_exact_parquet_row(qid)
    payload = _write_item_cache(qid, row, location, backend)
    item = _load_cached_item(qid)

    if item is None:
        raise RuntimeError(f"{qid}: cache reconstruction failed")

    _validate_item(qid, item)
    _ITEM_CACHE[qid] = item
    _upsert_hf_metadata(qid, item)

    if verbose:
        mb = float(payload.get("total_image_bytes", 0)) / (1024 * 1024)
        log(
            f"HF ready: {qid} | file={location['source_file']} | "
            f"row_group={location['row_group_idx']} | "
            f"local_row={location['local_row_in_group']} | "
            f"images={len(payload.get('image_columns', []))} | "
            f"selected_image_bytes={mb:.2f} MB"
        )

    return item

print("HF exact-row cache:", HF_ROW_CACHE_DIR)
print("HF metadata checkpoint:", HF_RECORDS_PATH)
print("Direct source Parquet retrieval ready.")
print("No Dataset Viewer endpoint will be used.")


Previous-output checkpoint candidates:
  /kaggle/input/datasets/sanasobhani/cotdataset/cot_pipeline_all_results.csv | valid= True | OK
VALIDATED PREVIOUS OUTPUT: /kaggle/input/datasets/sanasobhani/cotdataset/cot_pipeline_all_results.csv
Checkpoint status counts: {'ACCEPTED': 332, 'PENDING_STAGE1': 124, 'PENDING_STAGE1_API_ERROR': 72, 'REJECT_STAGE1_WRONG_ANSWER': 56, 'PENDING_STAGE1_INCOMPLETE': 9, 'REJECT_STAGE2_VALIDATION': 4, 'PENDING_STAGE2_API_ERROR': 3}
Dataset revision restored from checkpoint: 98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68
ID file: /kaggle/input/datasets/sanasobhani/cotdataset/correct_cot_similarity_pool_ids.txt
IDs: 600 | subjects: 30
Pinned MMMU revision: 98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68
ID SHA256: c7d44052ab7dc61cf1860eb490db125ff46156043d51b0439f6d80bce71572fd
Retrieval backend: pinned HF repository Parquet files via hf:// range reads
Dataset Viewer /rows: DISABLED


,split,samples
0,test,528
1,validation,61
2,dev,11


HF exact-row cache: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/hf_exact_row_cache
HF metadata checkpoint: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/selected_hf_records_metadata.csv
Direct source Parquet retrieval ready.
No Dataset Viewer endpoint will be used.


In [4]:
# Cell 4 — Image preparation, prompts, Google API, and parsers

# HF rows/images are retrieved lazily from the real pinned Parquet shards by get_item(qid).
# This cell only converts the cached exact images into API-ready content.

def json_safe(v):
    if v is None or isinstance(v, (str, int, float, bool)):
        return v
    try:
        return json.loads(json.dumps(v, ensure_ascii=False, default=str))
    except Exception:
        return str(v)

def get_item_images(item):
    return {
        i: item[f"image_{i}"]
        for i in range(1, 8)
        if item.get(f"image_{i}") is not None
    }

# ---------- Image blocks ----------
IMAGE_PLACEHOLDER_RE=re.compile(r"<image\s*(\d+)>",re.I)
_IMAGE_CACHE={}

def image_bytes(obj):
    if isinstance(obj,dict):
        if obj.get("bytes"): return bytes(obj["bytes"])
        if obj.get("path"): return Path(obj["path"]).read_bytes()
    if isinstance(obj,(bytes,bytearray)): return bytes(obj)
    if hasattr(obj,"save"):
        b=BytesIO(); obj.save(b,format="PNG"); return b.getvalue()
    raise TypeError(type(obj))

def rgb_white(img):
    img=ImageOps.exif_transpose(img)
    if img.mode in {"RGBA","LA"} or (img.mode=="P" and "transparency" in img.info):
        rgba=img.convert("RGBA")
        bg=PILImage.new("RGBA",rgba.size,(255,255,255,255))
        bg.alpha_composite(rgba)
        return bg.convert("RGB")
    return img.convert("RGB")

def encode_jpeg(img,q):
    b=BytesIO()
    img.save(b,format="JPEG",quality=int(q),optimize=True,subsampling=0)
    return b.getvalue()

def compress_image(raw):
    with PILImage.open(BytesIO(raw)) as src:
        img=rgb_white(src)
    if max(img.size)>API_IMAGE_MAX_SIDE:
        scale=API_IMAGE_MAX_SIDE/max(img.size)
        img=img.resize((max(1,round(img.width*scale)),max(1,round(img.height*scale))),PILImage.Resampling.LANCZOS)
    qualities=list(range(API_JPEG_QUALITY_START,API_JPEG_QUALITY_MIN-1,-4))
    best=None
    while True:
        for q in qualities:
            enc=encode_jpeg(img,q); best=enc
            if len(enc)<=API_IMAGE_TARGET_BYTES: return enc
        if max(img.size)<=640: return best
        img=img.resize((max(1,round(img.width*.9)),max(1,round(img.height*.9))),PILImage.Resampling.LANCZOS)

def image_block(obj):
    raw=image_bytes(obj)
    key=(hashlib.sha256(raw).hexdigest(),COMPRESS_IMAGES_FOR_API,API_IMAGE_MAX_SIDE,API_IMAGE_TARGET_BYTES)
    if key in _IMAGE_CACHE: return _IMAGE_CACHE[key]
    api_raw=compress_image(raw) if COMPRESS_IMAGES_FOR_API else raw
    mime="image/jpeg" if COMPRESS_IMAGES_FOR_API else "image/png"
    block={"type":"image","mime_type":mime,"data":base64.b64encode(api_raw).decode("ascii")}
    _IMAGE_CACHE[key]=block
    return block

def format_options(item):
    opts=parse_options(item["options"])
    return "\n".join(f"{l}. {o}" for l,o in zip(string.ascii_uppercase[:len(opts)],opts))

def interleave(item):
    q=str(item.get("question") or "")
    imgs=get_item_images(item)
    ms=list(IMAGE_PLACEHOLDER_RE.finditer(q))
    parts=[]
    if not ms:
        for i in sorted(imgs): parts.append(image_block(imgs[i]))
        if q: parts.append({"type":"text","text":q})
        return parts
    cursor=0; used=set()
    for m in ms:
        if q[cursor:m.start()]: parts.append({"type":"text","text":q[cursor:m.start()]})
        idx=int(m.group(1))
        if idx in imgs:
            parts.append(image_block(imgs[idx])); used.add(idx)
        else:
            parts.append({"type":"text","text":m.group(0)})
        cursor=m.end()
    if q[cursor:]: parts.append({"type":"text","text":q[cursor:]})
    for i in sorted(set(imgs)-used): parts.append(image_block(imgs[i]))
    return parts

# ---------- Exact prompts ----------
CALL1_PROMPT = """
Solve the following multimodal multiple-choice problem carefully.
Use the image(s), question, and answer choices to determine the answer.

Produce a concise but sufficient chain of reasoning.
- Express reasoning as atomic steps.
- Put exactly one substantive reasoning step on each separate line.
- Each step must contain one meaningful logical, mathematical, factual, or visual inference.
- Do not combine independent steps into one line.
- Do not split a trivial idea into multiple lines merely to create more steps.
- Use only the minimum number of steps necessary.
- Every visual claim must be supported by the image(s).
- Do not introduce unsupported facts.

Return exactly:
Reasoning:
Step 1: <first reasoning step>
Step 2: <second reasoning step>
...
Step N: <final reasoning step leading to the answer>
Answer: LETTER
""".strip()

CALL2_EXP = """
Validate the generated rationale exactly as written for use as a few-shot reasoning demonstration.

Check:
1. final answer matches the verified gold answer;
2. every substantive reasoning step is logically/factually correct;
3. visual claims are supported by the image(s);
4. it does not contradict the original dataset explanation. The explanation may be brief or incomplete; additional correct supported reasoning is allowed;
5. no unsupported fact is required for the conclusion;
6. reasoning genuinely supports the answer without a major misleading gap;
7. each numbered line is one substantive step; reject artificial fragmentation or improper collapsing.

Do NOT rewrite, repair, shorten, or expand the rationale.

Return exactly:
VERDICT: ACCEPT | REJECT
ANSWER_CORRECT: YES | NO
REASONING_CORRECT: YES | NO
VISUAL_GROUNDING_OK: YES | NO
CONTRADICTS_EXPLANATION: YES | NO
UNSUPPORTED_CLAIMS: YES | NO
MAJOR_REASONING_ERROR: YES | NO
ANSWER_SUPPORTED_BY_REASONING: YES | NO
STEP_FORMAT_OK: YES | NO
REASON: <brief reason>
""".strip()

CALL2_NOEXP = """
Validate the generated rationale exactly as written using the image(s), question, choices, and verified gold answer.

Check:
1. final answer matches gold;
2. every substantive reasoning step is correct;
3. visual claims are supported;
4. no unsupported fact is required for the conclusion;
5. reasoning genuinely supports the answer without a major misleading gap;
6. each numbered line is one substantive step and is not artificially fragmented.

Do NOT rewrite or repair the rationale.

Return exactly:
VERDICT: ACCEPT | REJECT
ANSWER_CORRECT: YES | NO
REASONING_CORRECT: YES | NO
VISUAL_GROUNDING_OK: YES | NO
UNSUPPORTED_CLAIMS: YES | NO
MAJOR_REASONING_ERROR: YES | NO
ANSWER_SUPPORTED_BY_REASONING: YES | NO
STEP_FORMAT_OK: YES | NO
REASON: <brief reason>
""".strip()

def build_call1(item):
    parts=interleave(item)
    parts.append({"type":"text","text":"\n\nAnswer choices:\n"+format_options(item)+"\n\n"+CALL1_PROMPT})
    return [{"type":"user_input","content":parts}]

def build_call2(item,generated):
    parts=interleave(item)
    gold=str(item.get("answer") or "").strip().upper()
    exp=str(item.get("explanation") or "").strip()
    has_exp=meaningful_explanation(exp)
    txt="\n\nAnswer choices:\n"+format_options(item)+f"\n\nVERIFIED GOLD ANSWER: {gold}\n"
    if has_exp: txt+="\nORIGINAL DATASET EXPLANATION:\n"+exp+"\n"
    txt+="\nGENERATED RATIONALE TO VALIDATE:\n"+generated.strip()+"\n\n"+(CALL2_EXP if has_exp else CALL2_NOEXP)
    parts.append({"type":"text","text":txt})
    return [{"type":"user_input","content":parts}], ("EXPLANATION_ASSISTED" if has_exp else "NO_EXPLANATION")

# ---------- Google API ----------
def objget(o,n,d=None):
    return o.get(n,d) if isinstance(o,dict) else getattr(o,n,d)

def extract_output(interaction):
    outs=[]; thoughts=[]
    for step in (objget(interaction,"steps",[]) or []):
        if objget(step,"type","")=="model_output":
            for part in (objget(step,"content",[]) or []):
                if objget(part,"type","")=="text" and objget(part,"text",""):
                    outs.append(str(objget(part,"text","")))
        elif objget(step,"type","")=="thought":
            for part in (objget(step,"summary",[]) or []):
                if objget(part,"type","")=="text" and objget(part,"text",""):
                    thoughts.append(str(objget(part,"text","")))
    content="\n".join(outs).strip() or str(objget(interaction,"output_text","") or "").strip()
    usage=objget(interaction,"usage",None)
    return {
        "content":content,
        "reasoning_content":"\n".join(thoughts).strip(),
        "prompt_tokens":objget(usage,"total_input_tokens",None),
        "completion_tokens":objget(usage,"total_output_tokens",None),
        "reasoning_tokens":objget(usage,"total_thought_tokens",None),
        "total_tokens":objget(usage,"total_tokens",None),
        "status":objget(interaction,"status",None),
        "interaction_id":objget(interaction,"id",None),
    }

def status_code(exc):
    for a in ("status_code","code"):
        try:
            v=getattr(exc,a,None)
            if v is not None:return int(v)
        except: pass
    m=re.search(r"\b(400|401|403|404|408|409|413|422|429|500|502|503|504)\b",str(exc))
    return int(m.group(1)) if m else None

def quota_error(status,text):
    low=str(text).lower()
    return status==429 or "resource_exhausted" in low or "rate limit" in low or ("quota" in low and "exceed" in low)

def api_call(qid,stage,model,api_input,temp,top_p,top_k,thinking_level,max_tokens):
    delay=INITIAL_BACKOFF_SECONDS; started=time.perf_counter()
    for attempt in range(1,MAX_API_RETRIES+1):
        try:
            if attempt == 1:
                print(
                    f"  {stage} config {qid}: "
                    f"thinking={thinking_level}, temp={temp}, top_p={top_p}, "
                    f"top_k={top_k}, max_tokens={max_tokens}, "
                    f"timeout_ms={API_HTTP_TIMEOUT_MS}"
                )
            inter=client.interactions.create(
                model=model,input=api_input,store=False,
                generation_config={
                    "temperature": temp,
                    "top_p": top_p,
                    "top_k": top_k,
                    "thinking_level": thinking_level,
                    "max_output_tokens": max_tokens,
                    "seed": MODEL_SEED,
                },
            )
            p=extract_output(inter); st=str(p.get("status") or "completed").lower()
            usable=st in {"completed","incomplete"} and bool(p["content"])
            return {"ok":usable,"quota_exhausted":False,"fatal":False,"http_status":200,"attempts":attempt,
                    "latency_s":time.perf_counter()-started,"finish_reason":st,
                    "error_type":"" if usable else f"INTERACTION_STATUS_{st.upper()}","error_message":"",
                    **p}
        except Exception as exc:
            s=status_code(exc); text=str(exc); low=text.lower(); quota=quota_error(s,text)
            retryable=quota or s in {408,409,500,502,503,504} or "timeout" in low or "temporarily unavailable" in low
            fatal=s in {400,401,403,404,413,422}
            if retryable and attempt<MAX_API_RETRIES:
                print(f"  {stage} transient error {qid} status={s}; retry in {delay:.1f}s")
                time.sleep(delay); delay=min(delay*2,MAX_BACKOFF_SECONDS); continue
            return {"ok":False,"quota_exhausted":quota,"fatal":fatal,"http_status":s,"attempts":attempt,
                    "latency_s":time.perf_counter()-started,"finish_reason":None,
                    "content":"","reasoning_content":"","prompt_tokens":None,"completion_tokens":None,
                    "reasoning_tokens":None,"total_tokens":None,"interaction_id":None,
                    "error_type":"QUOTA_OR_RATE_LIMIT_EXHAUSTED" if quota else (f"HTTP_{s}" if s else type(exc).__name__),
                    "error_message":text}

# ---------- Parsers ----------
ANSWER_RE=re.compile(r"Answer\s*:\s*([A-J])\b",re.I)
STEP_RE=re.compile(r"^\s*Step\s+(\d+)\s*:\s*(.+?)\s*$",re.I)

def parse_answer(text,item):
    matches=ANSWER_RE.findall(str(text or ""))
    if not matches:return None,"UNPARSEABLE"
    pred=matches[-1].upper()
    valid=set(string.ascii_uppercase[:len(parse_options(item["options"]))])
    return (pred,"OK") if pred in valid else (pred,"OUT_OF_RANGE")

def cot_info(text):
    text=str(text or "").strip()
    am=list(ANSWER_RE.finditer(text))
    reasoning=text[:am[-1].start()].strip() if am else text
    reasoning=re.sub(r"^\s*Reasoning\s*:\s*","",reasoning,flags=re.I).strip()
    nums=[]
    for line in reasoning.splitlines():
        m=STEP_RE.match(line)
        if m: nums.append(int(m.group(1)))
    return {
        "generated_cot":reasoning,
        "complexity_n_steps":len(nums),
        "step_numbers_sequential":nums==list(range(1,len(nums)+1)) if nums else False,
        "cot_word_count":len(reasoning.split()),
        "cot_char_count":len(reasoning),
    }

VAL_KEYS=["VERDICT","ANSWER_CORRECT","REASONING_CORRECT","VISUAL_GROUNDING_OK","CONTRADICTS_EXPLANATION",
          "UNSUPPORTED_CLAIMS","MAJOR_REASONING_ERROR","ANSWER_SUPPORTED_BY_REASONING","STEP_FORMAT_OK","REASON"]

def parse_validation(text,mode):
    out={k:None for k in VAL_KEYS}
    for raw in str(text or "").splitlines():
        line=raw.strip()
        for k in VAL_KEYS:
            if line.upper().startswith(k+":"):
                out[k]=line[len(k)+1:].strip(); break
    for k in VAL_KEYS[:-1]:
        if out[k] is not None: out[k]=out[k].upper()
    req=["VERDICT","ANSWER_CORRECT","REASONING_CORRECT","VISUAL_GROUNDING_OK","UNSUPPORTED_CLAIMS",
         "MAJOR_REASONING_ERROR","ANSWER_SUPPORTED_BY_REASONING","STEP_FORMAT_OK"]
    if mode=="EXPLANATION_ASSISTED": req.append("CONTRADICTS_EXPLANATION")
    parse_ok=all(out[k] is not None for k in req)
    strict=(parse_ok and out["VERDICT"]=="ACCEPT" and out["ANSWER_CORRECT"]=="YES"
            and out["REASONING_CORRECT"]=="YES" and out["VISUAL_GROUNDING_OK"]=="YES"
            and out["UNSUPPORTED_CLAIMS"]=="NO" and out["MAJOR_REASONING_ERROR"]=="NO"
            and out["ANSWER_SUPPORTED_BY_REASONING"]=="YES" and out["STEP_FORMAT_OK"]=="YES"
            and (mode!="EXPLANATION_ASSISTED" or out["CONTRADICTS_EXPLANATION"]=="NO"))
    out["PARSE_OK"]=parse_ok; out["STRICT_ACCEPT"]=strict
    return out

print("Prompt/API/parser utilities ready.")

Prompt/API/parser utilities ready.


In [5]:
# Cell 5 — Checkpoint state + two-stage processor

RESULT_COLUMNS = [
"request_index","question_id","subject","source_split",
"retrieval_backend","hf_source_file","hf_row_group_idx","hf_local_row_in_group","hf_global_split_offset","hf_retrieved_utc","image_sha256_json","hf_image_bytes",
"question","options_json","gold_answer",
"original_explanation","has_explanation","topic_difficulty","subfield","image_count","hf_nonimage_record_json",
"pipeline_status","accepted","rejected_stage1","stage1_reject_reason","rejected_stage2","stage2_reject_reason",
"call1_model","call1_temperature","call1_top_p","call1_top_k","call1_thinking_level","call1_seed","call1_max_output_tokens","call1_api_ok","call1_http_status","call1_attempts","call1_latency_s","call1_finish_reason",
"call1_prompt_tokens","call1_completion_tokens","call1_reasoning_tokens","call1_total_tokens","call1_interaction_id",
"call1_raw_response","call1_reasoning_content","generated_answer","answer_parse_status","stage1_answer_correct",
"generated_cot","complexity_n_steps","step_numbers_sequential","cot_word_count","cot_char_count","call1_error_type","call1_error_message",
"verification_mode","call2_model","call2_temperature","call2_top_p","call2_top_k","call2_thinking_level","call2_seed","call2_max_output_tokens","call2_api_ok","call2_http_status","call2_attempts","call2_latency_s","call2_finish_reason",
"call2_prompt_tokens","call2_completion_tokens","call2_reasoning_tokens","call2_total_tokens","call2_interaction_id",
"call2_raw_response","call2_reasoning_content","validator_verdict","validator_answer_correct","validator_reasoning_correct",
"validator_visual_grounding_ok","validator_contradicts_explanation","validator_unsupported_claims",
"validator_major_reasoning_error","validator_answer_supported_by_reasoning","validator_step_format_ok","validator_reason",
"validator_parse_ok","validator_strict_accept","call2_error_type","call2_error_message",
"dataset_revision","input_id_sha256","updated_utc"
]

TERMINAL={"ACCEPTED","REJECT_STAGE1_WRONG_ANSWER","REJECT_STAGE1_UNPARSEABLE_ANSWER","REJECT_STAGE2_VALIDATION"}

def default_row(qid):
    p = PARSED_BY_ID[qid]
    r = {c: None for c in RESULT_COLUMNS}
    r.update({
        "request_index": int(p["request_index"]),
        "question_id": qid,
        "subject": p["subject"],
        "source_split": p["split"],
        "pipeline_status": "PENDING_STAGE1",
        "accepted": False,
        "rejected_stage1": False,
        "rejected_stage2": False,
        "call1_model": SOLVER_MODEL,
        "call1_temperature": SOLVER_TEMPERATURE,
        "call1_top_p": SOLVER_TOP_P,
        "call1_top_k": SOLVER_TOP_K,
        "call1_thinking_level": SOLVER_THINKING_LEVEL,
        "call1_seed": MODEL_SEED,
        "call1_max_output_tokens": SOLVER_MAX_OUTPUT_TOKENS,
        "call2_model": VERIFIER_MODEL,
        "call2_temperature": VERIFIER_TEMPERATURE,
        "call2_top_p": VERIFIER_TOP_P,
        "call2_top_k": VERIFIER_TOP_K,
        "call2_thinking_level": VERIFIER_THINKING_LEVEL,
        "call2_seed": MODEL_SEED,
        "call2_max_output_tokens": VERIFIER_MAX_OUTPUT_TOKENS,
        "dataset_revision": DATASET_REVISION,
        "input_id_sha256": input_ids_sha256,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
    })
    return r

def hydrate_result_row_from_item(row, qid, item):
    """Copy exact HF source metadata into the persistent result row."""
    m = _metadata_from_item(qid, item)
    for k in [
        "request_index","question_id","subject","source_split",
        "retrieval_backend","hf_source_file","hf_row_group_idx","hf_local_row_in_group","hf_global_split_offset","hf_retrieved_utc",
        "image_sha256_json","hf_image_bytes",
        "question","options_json","gold_answer",
        "original_explanation","has_explanation","topic_difficulty",
        "subfield","image_count","hf_nonimage_record_json",
    ]:
        if k in m:
            row[k] = m[k]
    return row

_RESUME_BOOTSTRAPPED = False

def _bootstrap_resume_checkpoint_once():
    """
    On a fresh Kaggle session, copy the strictly validated external checkpoint
    into the active /kaggle/working output directory exactly once.
    """
    global _RESUME_BOOTSTRAPPED
    if _RESUME_BOOTSTRAPPED:
        return

    # Existing working checkpoint always takes precedence in the same session.
    if ALL_RESULTS_PATH.exists():
        local_info = validate_resume_checkpoint(ALL_RESULTS_PATH)
        if not local_info["valid"]:
            raise RuntimeError(
                "Existing working checkpoint failed strict validation: "
                + str(local_info["reason"])
            )
        print("Using existing working checkpoint:", ALL_RESULTS_PATH)
        _RESUME_BOOTSTRAPPED = True
        return

    if RESUME_CHECKPOINT_SOURCE is not None:
        src = Path(RESUME_CHECKPOINT_SOURCE)
        src_info = validate_resume_checkpoint(src)
        if not src_info["valid"]:
            raise RuntimeError(
                "External checkpoint became invalid before bootstrap: "
                + str(src_info["reason"])
            )

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, ALL_RESULTS_PATH)

        copied_info = validate_resume_checkpoint(ALL_RESULTS_PATH)
        if not copied_info["valid"]:
            raise RuntimeError(
                "Copied checkpoint failed post-copy validation: "
                + str(copied_info["reason"])
            )

        print("PREVIOUS OUTPUT IMPORTED AS ACTIVE CHECKPOINT:")
        print("  from:", src)
        print("  to:  ", ALL_RESULTS_PATH)

    _RESUME_BOOTSTRAPPED = True


def load_state():
    _bootstrap_resume_checkpoint_once()

    state={q:default_row(q) for q in REQUESTED_IDS}
    if ALL_RESULTS_PATH.exists():
        old=pd.read_csv(ALL_RESULTS_PATH)
        for _,row in old.iterrows():
            qid=str(row.get("question_id"))
            if qid in state:
                d={}
                for k,v in row.to_dict().items():
                    try:d[k]=None if pd.isna(v) else v
                    except:d[k]=v
                state[qid].update(d)

    # Hard post-load guards: terminal rows must remain terminal and the loaded
    # checkpoint must refer to exactly the current experiment.
    loaded_ids = {q for q, r in state.items() if r.get("question_id") == q}
    if loaded_ids != set(REQUESTED_IDS):
        raise RuntimeError("Loaded state ID universe mismatch.")

    return state

def _atomic_write_csv(df, path):
    """
    Write to a sibling temporary file, then atomically replace the active file.
    This protects the last complete checkpoint if the session dies during CSV
    serialization.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _atomic_write_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def save_all(state):
    """
    Persist the FULL cumulative state.

    This is intentionally an UPDATE/REWRITE of one row per requested ID,
    not a blind append. Blind appending would create duplicate IDs when a
    pending row advances from Stage 1 to Stage 2.
    """
    df=pd.DataFrame([state[q] for q in REQUESTED_IDS])
    for c in RESULT_COLUMNS:
        if c not in df: df[c]=None
    df["updated_utc"]=datetime.now(timezone.utc).isoformat()
    df=df[RESULT_COLUMNS]
    _atomic_write_csv(df, ALL_RESULTS_PATH)

    accepted=df[df["pipeline_status"].eq("ACCEPTED")].copy()
    _atomic_write_csv(accepted, ACCEPTED_ONLY_PATH)

    stats=[]
    for subject in sorted(parsed_ids_df["subject"].unique()):
        x=df[df["subject"].astype(str).eq(subject)]
        term=x[x["pipeline_status"].isin(TERMINAL)]
        acc=x[x["pipeline_status"].eq("ACCEPTED")]
        steps=pd.to_numeric(acc["complexity_n_steps"],errors="coerce").dropna()
        stats.append({
            "subject":subject,"requested":len(x),"terminal_processed":len(term),"accepted":len(acc),
            "rejected_stage1":int(x["pipeline_status"].astype(str).str.startswith("REJECT_STAGE1").sum()),
            "rejected_stage2":int(x["pipeline_status"].eq("REJECT_STAGE2_VALIDATION").sum()),
            "pending":int((~x["pipeline_status"].isin(TERMINAL)).sum()),
            "acceptance_rate_requested":len(acc)/len(x) if len(x) else None,
            "acceptance_rate_terminal":len(acc)/len(term) if len(term) else None,
            "mean_steps_accepted":float(steps.mean()) if len(steps) else None,
            "median_steps_accepted":float(steps.median()) if len(steps) else None,
            "min_steps_accepted":int(steps.min()) if len(steps) else None,
            "max_steps_accepted":int(steps.max()) if len(steps) else None,
        })
    statsdf=pd.DataFrame(stats); _atomic_write_csv(statsdf, SUBJECT_STATS_PATH)

    summary={
        "updated_utc":datetime.now(timezone.utc).isoformat(),
        "requested":len(df),"terminal":int(df["pipeline_status"].isin(TERMINAL).sum()),
        "accepted":int(df["pipeline_status"].eq("ACCEPTED").sum()),
        "rejected_stage1":int(df["pipeline_status"].astype(str).str.startswith("REJECT_STAGE1").sum()),
        "rejected_stage2":int(df["pipeline_status"].eq("REJECT_STAGE2_VALIDATION").sum()),
        "pending":int((~df["pipeline_status"].isin(TERMINAL)).sum()),
        "accepted_subjects":int(df.loc[df["pipeline_status"].eq("ACCEPTED"),"subject"].nunique()),
        "status_counts":{str(k):int(v) for k,v in df["pipeline_status"].value_counts(dropna=False).to_dict().items()},
    }
    _atomic_write_json(summary, SUMMARY_PATH)
    return df,statsdf,summary

def boolish(v): return v is True or str(v).strip().lower()=="true"
def has_stage1(row):
    return boolish(row.get("call1_api_ok")) and boolish(row.get("stage1_answer_correct")) and bool(str(row.get("call1_raw_response") or "").strip())

def put_api(row,prefix,res):
    mapping={"api_ok":"ok","http_status":"http_status","attempts":"attempts","latency_s":"latency_s",
             "finish_reason":"finish_reason","prompt_tokens":"prompt_tokens","completion_tokens":"completion_tokens",
             "reasoning_tokens":"reasoning_tokens","total_tokens":"total_tokens","interaction_id":"interaction_id",
             "raw_response":"content","reasoning_content":"reasoning_content","error_type":"error_type","error_message":"error_message"}
    for a,b in mapping.items(): row[f"{prefix}_{a}"]=res.get(b)

class QuotaStop(RuntimeError):pass
class FatalStop(RuntimeError):pass

def print_resume_audit(state):
    audit_df = pd.DataFrame([state[q] for q in REQUESTED_IDS])
    counts = audit_df["pipeline_status"].fillna("NaN").value_counts()
    terminal_n = int(audit_df["pipeline_status"].isin(TERMINAL).sum())
    accepted_n = int(audit_df["pipeline_status"].eq("ACCEPTED").sum())
    pending_n = int(len(audit_df) - terminal_n)

    print("\n" + "="*100)
    print("RESUME STATE AUDIT")
    print("="*100)
    print("Requested:", len(audit_df))
    print("Terminal (will be skipped):", terminal_n)
    print("Accepted already preserved:", accepted_n)
    print("Pending/retryable:", pending_n)
    print("Status counts:")
    print(counts.to_string())

    # Exact stage workload.
    stage2_retry = int(audit_df["pipeline_status"].isin({
        "PENDING_STAGE2",
        "PENDING_STAGE2_API_ERROR",
        "PENDING_STAGE2_QUOTA",
        "PENDING_STAGE2_FATAL_API",
    }).sum())
    untouched = int(audit_df["pipeline_status"].eq("PENDING_STAGE1").sum())
    print("Untouched Stage-1 rows:", untouched)
    print("Stage-2-only retry rows:", stage2_retry)
    print("="*100 + "\n")


def process_one(qid,state,verbose=True):
    row=state[qid]
    if str(row.get("pipeline_status")) in TERMINAL:
        if verbose: print(qid,"SKIP",row["pipeline_status"])
        return

    # LAZY exact HF retrieval happens only now, for this one sample.
    item=get_item(qid, verbose=verbose)
    hydrate_result_row_from_item(row, qid, item)
    state[qid]=row
    save_all(state)

    # CALL 1
    if not has_stage1(row):
        if verbose:
            print("\n"+"="*100); print("CALL 1",qid,"|",item["_source_config"],"| gold",item["answer"]); print("="*100)
        r1=api_call(
            qid, "CALL1", SOLVER_MODEL, build_call1(item),
            SOLVER_TEMPERATURE, SOLVER_TOP_P, SOLVER_TOP_K,
            SOLVER_THINKING_LEVEL, SOLVER_MAX_OUTPUT_TOKENS
        )
        put_api(row,"call1",r1); state[qid]=row; save_all(state)

        if not r1["ok"]:
            if r1["quota_exhausted"]:
                row["pipeline_status"]="PENDING_STAGE1_QUOTA"; save_all(state); raise QuotaStop(qid)
            if r1["fatal"]:
                row["pipeline_status"]="PENDING_STAGE1_FATAL_API"; save_all(state); raise FatalStop(r1["error_message"])
            row["pipeline_status"]="PENDING_STAGE1_API_ERROR"; save_all(state); return

        pred,pstatus=parse_answer(r1["content"],item)
        gold=str(item["answer"]).strip().upper()
        ci=cot_info(r1["content"])
        row.update({"generated_answer":pred,"answer_parse_status":pstatus,"stage1_answer_correct":pred==gold if pred else False,**ci})

        if verbose:
            print(r1["content"])
            print("Parsed:",pred,"Gold:",gold,"Correct:",pred==gold,"Steps:",ci["complexity_n_steps"])

        if pred is None:
            if str(r1.get("finish_reason") or "").lower()=="incomplete":
                row["pipeline_status"]="PENDING_STAGE1_INCOMPLETE"; save_all(state); return
            row.update({"pipeline_status":"REJECT_STAGE1_UNPARSEABLE_ANSWER","rejected_stage1":True,
                        "stage1_reject_reason":"UNPARSEABLE_FINAL_ANSWER","accepted":False})
            save_all(state); return

        if pred!=gold:
            row.update({"pipeline_status":"REJECT_STAGE1_WRONG_ANSWER","rejected_stage1":True,
                        "stage1_reject_reason":f"WRONG_{pred}_VS_{gold}","accepted":False})
            save_all(state)
            if verbose: print("REJECT STAGE 1 — CALL 2 NOT MADE")
            return

        row["pipeline_status"]="PENDING_STAGE2"; save_all(state)

    # CALL 2
    generated=str(row.get("call1_raw_response") or "").strip()
    inp,mode=build_call2(item,generated); row["verification_mode"]=mode

    if verbose:
        print("\n"+"="*100); print("CALL 2",qid,"| mode",mode); print("="*100)

    r2=api_call(
        qid, "CALL2", VERIFIER_MODEL, inp,
        VERIFIER_TEMPERATURE, VERIFIER_TOP_P, VERIFIER_TOP_K,
        VERIFIER_THINKING_LEVEL, VERIFIER_MAX_OUTPUT_TOKENS
    )
    put_api(row,"call2",r2); save_all(state)

    if not r2["ok"]:
        if r2["quota_exhausted"]:
            row["pipeline_status"]="PENDING_STAGE2_QUOTA"; save_all(state); raise QuotaStop(qid)
        if r2["fatal"]:
            row["pipeline_status"]="PENDING_STAGE2_FATAL_API"; save_all(state); raise FatalStop(r2["error_message"])
        row["pipeline_status"]="PENDING_STAGE2_API_ERROR"; save_all(state); return

    v=parse_validation(r2["content"],mode)
    row.update({
        "validator_verdict":v["VERDICT"],"validator_answer_correct":v["ANSWER_CORRECT"],
        "validator_reasoning_correct":v["REASONING_CORRECT"],"validator_visual_grounding_ok":v["VISUAL_GROUNDING_OK"],
        "validator_contradicts_explanation":v["CONTRADICTS_EXPLANATION"],"validator_unsupported_claims":v["UNSUPPORTED_CLAIMS"],
        "validator_major_reasoning_error":v["MAJOR_REASONING_ERROR"],
        "validator_answer_supported_by_reasoning":v["ANSWER_SUPPORTED_BY_REASONING"],
        "validator_step_format_ok":v["STEP_FORMAT_OK"],"validator_reason":v["REASON"],
        "validator_parse_ok":v["PARSE_OK"],"validator_strict_accept":v["STRICT_ACCEPT"],
    })

    if verbose:
        print(r2["content"]); print("Strict accept:",v["STRICT_ACCEPT"],"| reason:",v["REASON"])

    if not v["PARSE_OK"]:
        row["pipeline_status"]="PENDING_STAGE2_VALIDATION_PARSE"; save_all(state); return

    if v["STRICT_ACCEPT"]:
        row.update({"pipeline_status":"ACCEPTED","accepted":True,"rejected_stage2":False,"stage2_reject_reason":None})
    else:
        bits=[]
        expected={"VERDICT":"ACCEPT","ANSWER_CORRECT":"YES","REASONING_CORRECT":"YES","VISUAL_GROUNDING_OK":"YES",
                  "UNSUPPORTED_CLAIMS":"NO","MAJOR_REASONING_ERROR":"NO","ANSWER_SUPPORTED_BY_REASONING":"YES","STEP_FORMAT_OK":"YES"}
        for k,e in expected.items():
            if v[k]!=e: bits.append(f"{k}!={e}")
        if mode=="EXPLANATION_ASSISTED" and v["CONTRADICTS_EXPLANATION"]!="NO":
            bits.append("CONTRADICTS_EXPLANATION!=NO")
        row.update({"pipeline_status":"REJECT_STAGE2_VALIDATION","accepted":False,"rejected_stage2":True,
                    "stage2_reject_reason":"; ".join(bits) or "VALIDATOR_REJECT"})
    save_all(state)

state=load_state()
results_df,subject_stats_df,summary_dict=save_all(state)
print("Checkpoint ready:",ALL_RESULTS_PATH)
display(results_df["pipeline_status"].value_counts().rename_axis("status").reset_index(name="samples"))

PREVIOUS OUTPUT IMPORTED AS ACTIVE CHECKPOINT:
  from: /kaggle/input/datasets/sanasobhani/cotdataset/cot_pipeline_all_results.csv
  to:   /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_all_results.csv
Checkpoint ready: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_all_results.csv


,status,samples
0,ACCEPTED,332
1,PENDING_STAGE1,124
2,PENDING_STAGE1_API_ERROR,72
3,REJECT_STAGE1_WRONG_ANSWER,56
4,PENDING_STAGE1_INCOMPLETE,9
5,REJECT_STAGE2_VALIDATION,4
6,PENDING_STAGE2_API_ERROR,3


## First run: one-sample smoke test

Run only the next cell first.

Expected configuration printed for the sample:

```text
CALL1:
thinking=high
temperature=1.0
top_p=0.95
top_k=64
max_tokens=4096
timeout_ms=900000

CALL2 (only if Call 1 answer is correct):
thinking=high
temperature=0.0
top_p=1.0
top_k=64
max_tokens=2048
timeout_ms=900000
```

This notebook uses a new output directory, so older CoTs generated under the
previous decoding/thinking configuration are not mixed into this experiment.


In [6]:
# Cell 6 — ONE-SAMPLE SMOKE TEST

SMOKE_ID=REQUESTED_IDS[0]
state=load_state()
# Immediately materialize a complete cumulative working copy:
# previous state + current pending rows, before any new API request.
save_all(state)
print_resume_audit(state)

print("SMOKE ID:",SMOKE_ID)
print("Fixed retrieval: direct pinned HF Parquet row-group access; Dataset Viewer disabled.")
try:
    process_one(SMOKE_ID,state,verbose=True)
except QuotaStop:
    print("QUOTA STOP. Rotate API secret and rerun this cell.")
except FatalStop as e:
    print("FATAL:",e); raise

results_df,subject_stats_df,summary_dict=save_all(state)

display(results_df.loc[
    results_df["question_id"].astype(str).eq(SMOKE_ID),
    ["question_id","subject","gold_answer","has_explanation","pipeline_status",
     "generated_answer","generated_cot","complexity_n_steps","step_numbers_sequential",
     "verification_mode","validator_verdict","validator_strict_accept",
     "stage1_reject_reason","stage2_reject_reason","validator_reason"]
])


RESUME STATE AUDIT
Requested: 600
Terminal (will be skipped): 392
Accepted already preserved: 332
Pending/retryable: 208
Status counts:
pipeline_status
ACCEPTED                      332
PENDING_STAGE1                124
PENDING_STAGE1_API_ERROR       72
REJECT_STAGE1_WRONG_ANSWER     56
PENDING_STAGE1_INCOMPLETE       9
REJECT_STAGE2_VALIDATION        4
PENDING_STAGE2_API_ERROR        3
Untouched Stage-1 rows: 124
Stage-2-only retry rows: 3

SMOKE ID: test_Accounting_22
Fixed retrieval: direct pinned HF Parquet row-group access; Dataset Viewer disabled.
test_Accounting_22 SKIP ACCEPTED


,question_id,subject,gold_answer,has_explanation,pipeline_status,generated_answer,generated_cot,complexity_n_steps,step_numbers_sequential,verification_mode,validator_verdict,validator_strict_accept,stage1_reject_reason,stage2_reject_reason,validator_reason
0,test_Accounting_22,Accounting,C,True,ACCEPTED,C,"Step 1: Identify the returns and probabilities for investing in equities from the table: there is a 0.6 probability of a $50,000 return and a 0.4 probability of a -$30,000 retu...",4.0,True,EXPLANATION_ASSISTED,ACCEPT,True,None,None,"The rationale correctly identifies the data from the table, calculates the expected return for equities and the risk-free asset, and determines the risk premium by finding the ..."


In [7]:
# Cell 7 — API KEY ROTATION / RESUME
#
# OPTION A — simplest:
# Replace the VALUE of the existing Kaggle Secret `GEMINI_API_KEY`
# with a new API key, then rerun Cell 2.
#
# OPTION B — use another Secret alias:
# 1) create `GEMINI_API_KEY_2` in Kaggle Secrets
# 2) uncomment and run:
#
# connect_google_client("GEMINI_API_KEY_2")
#
# Then rerun the FULL RUN cell.
#
# The checkpoint logic skips completed stages. For example:
# - if Call 1 succeeded but Call 2 hit quota,
# - Call 1 is NOT repeated,
# - the sample resumes directly at Call 2.

print("Current secret alias:", ACTIVE_API_SECRET)
print("Checkpoint:", ALL_RESULTS_PATH)


Current secret alias: GEMINI_API_KEY
Checkpoint: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_all_results.csv


## How to continue a previous Kaggle run

Attach these **two inputs** to the new Kaggle notebook:

1. The **exact same 600-ID TXT** from the original experiment.
2. The latest previous:
   `cot_pipeline_all_results.csv`

You do **not** need to upload the old `accepted_only`, stats, or summary files.
They are deterministically regenerated from `cot_pipeline_all_results.csv`.

Leave:

```python
PREVIOUS_RESULTS_CSV = None
```

The notebook automatically searches `/kaggle/input/**/cot_pipeline_all_results*.csv`.

Before making a new API call it must print:

```text
VALIDATED PREVIOUS OUTPUT: ...
PREVIOUS OUTPUT IMPORTED AS ACTIVE CHECKPOINT:
  from: /kaggle/input/...
  to:   /kaggle/working/.../cot_pipeline_all_results.csv

RESUME STATE AUDIT
...
```

### Resume semantics

- `ACCEPTED` → skipped.
- Stage-1 wrong-answer rejection → skipped.
- Stage-2 validation rejection → skipped.
- Stage-2 API/quota error with successful Call 1 → Call 1 is reused; only Call 2 is retried.
- Stage-1 API/incomplete error → Call 1 is retried.
- untouched row → starts normally.

### Cumulative outputs

The files under `/kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/`
always represent the **entire current state**, not only the new session:

- `cot_pipeline_all_results.csv`
- `cot_pipeline_accepted_only.csv`
- `cot_pipeline_subject_stats.csv`
- `cot_pipeline_summary.json`

So after the new Version finishes or stops, download the newest
`cot_pipeline_all_results.csv`; that newest file becomes the checkpoint for
the next continuation.

The old input CSV is never modified because `/kaggle/input` is read-only.


In [8]:
# Cell 8 — FULL 600-ID RUN (direct pinned Parquet retrieval; safe to rerun/resume)

def run_full_batch():
    state=load_state()
    # Immediately regenerate all cumulative output files before new inference.
    save_all(state)
    print_resume_audit(state)
    for i,qid in enumerate(REQUESTED_IDS,1):
        status=str(state[qid].get("pipeline_status") or "")
        print("\n"+"#"*100)
        print(f"{i:03d}/{len(REQUESTED_IDS)} | {qid} | {status}")
        print("#"*100)

        if status in TERMINAL:
            print("SKIP terminal")
            continue

        try:
            process_one(qid,state,verbose=True)
        except QuotaStop:
            save_all(state)
            print("\nBATCH PAUSED — quota/rate limit.")
            print("Rotate the Kaggle Secret in Cell 7, reconnect, then rerun THIS cell.")
            print("Checkpoint resumes the exact pending stage.")
            return
        except FatalStop as e:
            save_all(state)
            print("\nBATCH STOPPED — fatal API error:",e)
            return

        if REQUEST_DELAY_SECONDS>0:
            time.sleep(REQUEST_DELAY_SECONDS)

    df,stats,summary=save_all(state)
    print("FULL RUN FINISHED")
    display(df["pipeline_status"].value_counts().rename_axis("status").reset_index(name="samples"))

run_full_batch()


RESUME STATE AUDIT
Requested: 600
Terminal (will be skipped): 392
Accepted already preserved: 332
Pending/retryable: 208
Status counts:
pipeline_status
ACCEPTED                      332
PENDING_STAGE1                124
PENDING_STAGE1_API_ERROR       72
REJECT_STAGE1_WRONG_ANSWER     56
PENDING_STAGE1_INCOMPLETE       9
REJECT_STAGE2_VALIDATION        4
PENDING_STAGE2_API_ERROR        3
Untouched Stage-1 rows: 124
Stage-2-only retry rows: 3


####################################################################################################
001/600 | test_Accounting_22 | ACCEPTED
####################################################################################################
SKIP terminal

####################################################################################################
002/600 | test_Accounting_285 | PENDING_STAGE1_API_ERROR
####################################################################################################
[20:26:57] HF Parquet exact fetch: t

,status,samples
0,ACCEPTED,426
1,PENDING_STAGE1_API_ERROR,90
2,REJECT_STAGE1_WRONG_ANSWER,66
3,PENDING_STAGE1_INCOMPLETE,10
4,PENDING_STAGE2_API_ERROR,4
5,REJECT_STAGE2_VALIDATION,4


In [9]:
# Cell 9 — Paper/thesis-ready summary

state=load_state()
results_df,subject_stats_df,summary_dict=save_all(state)

requested=len(results_df)
terminal=int(results_df["pipeline_status"].isin(TERMINAL).sum())
accepted=int(results_df["pipeline_status"].eq("ACCEPTED").sum())
rej1=int(results_df["pipeline_status"].astype(str).str.startswith("REJECT_STAGE1").sum())
rej2=int(results_df["pipeline_status"].eq("REJECT_STAGE2_VALIDATION").sum())
pending=requested-terminal

print("="*100)
print("TWO-STAGE COT PIPELINE SUMMARY")
print("="*100)
print("Requested:",requested)
print("Terminal:",terminal)
print("Accepted:",accepted)
print("Rejected Stage 1:",rej1)
print("Rejected Stage 2:",rej2)
print("Pending/API:",pending)
print("Subjects with >=1 accepted:",results_df.loc[results_df["pipeline_status"].eq("ACCEPTED"),"subject"].nunique())
if terminal: print("Acceptance / terminal:",f"{accepted/terminal:.1%}")
if requested: print("Acceptance / requested:",f"{accepted/requested:.1%}")

acc=results_df[results_df["pipeline_status"].eq("ACCEPTED")]
if len(acc):
    steps=pd.to_numeric(acc["complexity_n_steps"],errors="coerce").dropna()
    print("\nAccepted complexity steps:")
    print("mean=",round(float(steps.mean()),2) if len(steps) else None,
          "median=",round(float(steps.median()),2) if len(steps) else None,
          "min=",int(steps.min()) if len(steps) else None,
          "max=",int(steps.max()) if len(steps) else None)

print("\nSubject statistics:")
display(subject_stats_df)

print("\nSaved:")
print("Manifest:",MANIFEST_PATH)
print("HF metadata:",HF_RECORDS_PATH)
print("All results:",ALL_RESULTS_PATH)
print("Accepted only:",ACCEPTED_ONLY_PATH)
print("Subject stats:",SUBJECT_STATS_PATH)
print("Summary:",SUMMARY_PATH)

TWO-STAGE COT PIPELINE SUMMARY
Requested: 600
Terminal: 496
Accepted: 426
Rejected Stage 1: 66
Rejected Stage 2: 4
Pending/API: 104
Subjects with >=1 accepted: 30
Acceptance / terminal: 85.9%
Acceptance / requested: 71.0%

Accepted complexity steps:
mean= 5.58 median= 5.0 min= 3 max= 12

Subject statistics:


,subject,requested,terminal_processed,accepted,rejected_stage1,rejected_stage2,pending,acceptance_rate_requested,acceptance_rate_terminal,mean_steps_accepted,median_steps_accepted,min_steps_accepted,max_steps_accepted
0,Accounting,20,17,14,3,0,3,0.70,0.823529,6.928571,7.0,4,10
1,Agriculture,20,20,14,6,0,0,0.70,0.700000,4.642857,5.0,3,6
2,Architecture_and_Engineering,20,4,4,0,0,16,0.20,1.000000,6.250000,5.5,4,10
3,Art,20,20,17,3,0,0,0.85,0.850000,4.529412,5.0,3,5
4,Art_Theory,20,20,19,1,0,0,0.95,0.950000,4.631579,5.0,4,6
5,Basic_Medical_Science,20,20,17,3,0,0,0.85,0.850000,5.588235,5.0,4,8
6,Biology,20,20,17,3,0,0,0.85,0.850000,4.705882,5.0,3,6
7,Chemistry,20,18,18,0,0,2,0.90,1.000000,6.555556,6.0,5,11
8,Clinical_Medicine,20,18,8,10,0,2,0.40,0.444444,5.125000,5.0,4,6
9,Computer_Science,20,19,16,2,1,1,0.80,0.842105,6.125000,6.0,4,9



Saved:
Manifest: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/manifest.json
HF metadata: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/selected_hf_records_metadata.csv
All results: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_all_results.csv
Accepted only: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_accepted_only.csv
Subject stats: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_subject_stats.csv
Summary: /kaggle/working/mmmu_correct_cot_gemma4_high_thinking_v4/cot_pipeline_summary.json


### Downstream use and methodological record

Use `cot_pipeline_accepted_only.csv` as the clean positive-CoT pool.

This v4 run records the inference configuration per row and in `manifest.json`:

- Call 1: Gemma 4 31B, thinking=high, temperature=1.0, top_p=0.95, top_k=64.
- Call 2: Gemma 4 31B, thinking=high, temperature=0.0, top_p=1.0, top_k=64.
- Seed: 42.
- Client-side timeout: 900000 ms (15 minutes).
- Wrong Call-1 answers are rejected without making Call 2.
- Call 2 never rewrites the CoT; it validates it.
- Explanation, when present, is visible only to Call 2.
- `complexity_n_steps` is retained for Complexity-Based Prompting.
- Exact source Parquet, row-group location, image hashes, and pinned dataset
  revision are retained for reproducibility.
